# Layer 8 — Research & Hypothesis Engine

**Role:** The analytical research layer.
Layer 8 does not collect or generate new field data. It reads the growing
`layer7_test_history.jsonl` archive and searches for reproducible patterns,
correlations, and structural relationships across snapshots.

**Architecture:** Snapshot-based, 4 runs per day at fixed times (CEST / UTC+2):

| Slot | Time | Carnegie Context |
|---|---|---|
| Morning | 06:30 CEST | Carnegie trough — baseline |
| Midday | 12:30 CEST | Carnegie rising |
| Evening | 18:30 CEST | Carnegie peak — activation |
| Night | 22:30 CEST | Carnegie declining |

Day-pair analysis uses Morning (06:30) as baseline and Evening (18:30) as
activation reference. ΔL3, ΔL5, ΔL6 measure the daily activation between
these two anchors.

**Inputs:** `layer7_test_history.jsonl`

**Outputs:**
- `layer8_test_state.json` — machine-readable full analysis result
- `layer8_test_report.md` — human-readable summary report
- `data/research/test_hypothesis_registry.json` — persistent hypothesis registry
- `data/research/test_hypothesis_candidates/*.json` — one file per hypothesis candidate

## 0. Setup

In [1]:
import json, math
from datetime import datetime, timedelta, timezone
from collections import Counter, defaultdict
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

HISTORY_FILE   = '../data/history/layer7_test_history.jsonl'
STATE_FILE     = '../data/states/layer8_test_state.json'
REPORT_FILE    = '../research/layer8_test_report.md'

CEST           = timedelta(hours=2)
RUN_TS         = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
ENGINE_VERSION = '1.0'

print(f'Layer 8 Engine {ENGINE_VERSION}  |  Run: {RUN_TS}')

Layer 8 Engine 1.0  |  Run: 2026-06-14T21:56:29.419620Z


## 1. Load & Deduplicate History

Duplicate snapshots (with the same timestamp) result from re-runs of the same Layer 7 session. We retain only the first one.

In [2]:
history = []
with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            history.append(json.loads(line))

# Deduplicate by timestamp
seen = set()
snaps = []
for s in history:
    ts = s.get('timestamp')
    if ts and ts not in seen:
        seen.add(ts)
        snaps.append(s)

# Sort chronologically
snaps.sort(key=lambda s: s['timestamp'])

# Slot annotation — 4 slots per day (CEST)
# 06:30 = morning | 12:30 = midday | 18:30 = evening | 22:30 = night
def classify_slot(hour):
    if   4 <= hour < 10:  return 'morning'   # 06:30
    elif 10 <= hour < 16: return 'midday'    # 12:30
    elif 16 <= hour < 21: return 'evening'   # 18:30
    else:                 return 'night'     # 22:30

for s in snaps:
    t_utc  = datetime.fromisoformat(s['timestamp'].replace('Z',''))
    t_cest = t_utc + CEST
    s['_cest'] = t_cest
    s['_date'] = t_cest.strftime('%Y-%m-%d')
    s['_slot'] = classify_slot(t_cest.hour)

print(f'Snapshots loaded:         {len(history)}')
print(f'After dedup:              {len(snaps)}')
print(f'Time range:               {snaps[0]["_cest"].strftime("%Y-%m-%d %H:%M")} – {snaps[-1]["_cest"].strftime("%Y-%m-%d %H:%M")} CEST')
print(f'Morning snapshots:        {sum(1 for s in snaps if s["_slot"] == "morning")}')
print(f'Midday snapshots:         {sum(1 for s in snaps if s["_slot"] == "midday")}')
print(f'Evening snapshots:        {sum(1 for s in snaps if s["_slot"] == "evening")}')
print(f'Night snapshots:          {sum(1 for s in snaps if s["_slot"] == "night")}')


Snapshots loaded:         21
After dedup:              21
Time range:               2026-05-10 11:01 – 2026-05-14 14:14 CEST
Morning snapshots:        4
Midday snapshots:         10
Evening snapshots:        4
Night snapshots:          3


## 2. FORMING DAY QUADS

Each day should have all 4 slots (morning / midday / evening / night).
The central analysis unit remains the morning–evening pair:
ΔL3, ΔL5, and ΔOperator are only meaningful between these two anchors.
Midday and night slots are retained for trend and transition analysis.

In [3]:
by_date = defaultdict(dict)
for s in snaps:
    # if multiple snapshots exist in the same slot: keep the first (stable baseline)
    if s['_slot'] not in by_date[s['_date']]:
        by_date[s['_date']][s['_slot']] = s

day_quads = []
for date in sorted(by_date.keys()):
    morning = by_date[date].get('morning')
    midday  = by_date[date].get('midday')
    evening = by_date[date].get('evening')
    night   = by_date[date].get('night')
    day_quads.append({
        'date':          date,
        'morning':       morning,
        'midday':        midday,
        'evening':       evening,
        'night':         night,
        'complete':      bool(morning and evening),
        'complete_full': bool(morning and midday and evening and night),
    })

complete_pairs = [d for d in day_quads if d['complete']]
complete_full  = [d for d in day_quads if d['complete_full']]
n_pairs        = len(complete_pairs)

print(f'Days total:               {len(day_quads)}')
print(f'Complete core pairs:      {n_pairs}  (morning + evening)')
print(f'Complete full quads:      {len(complete_full)}  (all 4 slots)')
print()
for d in day_quads:
    slots = []
    for slot in ['morning', 'midday', 'evening', 'night']:
        s = d[slot]
        slots.append(f'{slot}={s["_cest"].strftime("%H:%M")}' if s else f'{slot}=—')
    core_icon = '✓' if d['complete']      else '✗'
    full_icon = '✓' if d['complete_full'] else '✗'
    print(f'  core={core_icon} full={full_icon}  {d["date"]}  {" | ".join(slots)}')

Days total:               5
Complete core pairs:      3  (morning + evening)
Complete full quads:      3  (all 4 slots)

  core=✗ full=✗  2026-05-10  morning=— | midday=11:01 | evening=19:31 | night=—
  core=✓ full=✓  2026-05-11  morning=07:16 | midday=10:05 | evening=20:20 | night=23:53
  core=✓ full=✓  2026-05-12  morning=09:27 | midday=14:26 | evening=20:24 | night=23:53
  core=✓ full=✓  2026-05-13  morning=09:40 | midday=14:31 | evening=20:28 | night=23:57
  core=✗ full=✗  2026-05-14  morning=09:33 | midday=12:16 | evening=— | night=—


## 3. State and Layer Profile

Frequency of system states, range of layer scores, dominant layers.

In [4]:
LAYERS = ['L0_external_drivers', 'L1_planetary_body', 'L2_surface_zone',
          'L3_atmosphere', 'L4_ionosphere', 'L5_global_electric_circuit',
          'L6_resonance_field']

def lscore(snap, lname):
    return snap['layers'].get(lname, {}).get('score')

# State frequency
state_counts = Counter(s['system_state'] for s in snaps)
state_freq = {state: {'count': c, 'pct': round(c/len(snaps)*100, 1)}
              for state, c in state_counts.most_common()}

# Layer statistics
layer_stats = {}
for lname in LAYERS:
    vals = [lscore(s, lname) for s in snaps]
    vals = [v for v in vals if v is not None]
    if vals:
        layer_stats[lname] = {
            'mean':  round(float(np.mean(vals)),  4),
            'std':   round(float(np.std(vals)),   4),
            'min':   round(float(np.min(vals)),   4),
            'max':   round(float(np.max(vals)),   4),
            'range': round(float(np.max(vals) - np.min(vals)), 4),
        }

# Dominance frequency
dom_counts = Counter(s['dominance']['dominant_layer'] for s in snaps if s.get('dominance'))

# Output
print('SYSTEM STATES')
print('=' * 65)
for state, stats in state_freq.items():
    bar = '█' * stats['count']
    print(f'  {state:<38} {bar} {stats["count"]:>2}× ({stats["pct"]:.0f}%)')

print()
print('LAYER SCORE PROFILE')
print('=' * 65)
print(f'  {"Layer":<32} {"mean":>6} {"std":>6} {"min":>6} {"max":>6}')
for lname, st in layer_stats.items():
    print(f'  {lname:<32} {st["mean"]:>6.3f} {st["std"]:>6.3f} {st["min"]:>6.3f} {st["max"]:>6.3f}')

print()
print('DOMINANCE')
print('=' * 65)
for layer, c in dom_counts.most_common():
    print(f'  {layer:<32} {c:>2}× ({c/len(snaps)*100:.0f}%)')

SYSTEM STATES
  seasonal_transition_state              ███████████ 11× (52%)
  cavity_condition_shift_state           ███████  7× (33%)
  anomalous_resonance_state              ███  3× (14%)

LAYER SCORE PROFILE
  Layer                              mean    std    min    max
  L0_external_drivers               0.173  0.027  0.131  0.232
  L1_planetary_body                 0.319  0.026  0.245  0.371
  L2_surface_zone                   0.440  0.099  0.344  0.604
  L3_atmosphere                     0.151  0.023  0.108  0.207
  L4_ionosphere                     0.306  0.026  0.252  0.360
  L5_global_electric_circuit        0.288  0.062  0.227  0.479
  L6_resonance_field                0.288  0.018  0.251  0.331

DOMINANCE
  L2_surface_zone                  15× (71%)
  L5_global_electric_circuit        4× (19%)
  L1_planetary_body                 2× (10%)


## 4. Morning vs Evening — ΔL3 as Activation Indicator

ΔL3 = L3 score (evening 18:30) − L3 score (morning 06:30).
ΔL3 measures how strongly the atmosphere activated during the day.

The midday slot (12:30) serves as an intermediate check:
a rising ΔL3 between morning and midday may indicate early activation.
The night slot (22:30) shows whether activation persisted after the Carnegie peak.

**Hypothesis:** ΔL3 > ~0.035 is a prerequisite for `anomalous_resonance_state`.

In [5]:
dl3_data = []
for d in complete_pairs:
    m, e = d['morning'], d['evening']
    mid  = d.get('midday')
    ngt  = d.get('night')

    dl3 = lscore(e, 'L3_atmosphere') - lscore(m, 'L3_atmosphere')
    dl5 = lscore(e, 'L5_global_electric_circuit') - lscore(m, 'L5_global_electric_circuit')
    dl6 = lscore(e, 'L6_resonance_field')         - lscore(m, 'L6_resonance_field')

    # midday: early activation check (morning → midday)
    dl3_mid = round(lscore(mid, 'L3_atmosphere') - lscore(m, 'L3_atmosphere'), 4) \
              if mid and lscore(mid, 'L3_atmosphere') is not None else None

    # night: persistence check (evening → night)
    dl3_ngt = round(lscore(ngt, 'L3_atmosphere') - lscore(e, 'L3_atmosphere'), 4) \
              if ngt and lscore(ngt, 'L3_atmosphere') is not None else None

    dl3_data.append({
        'date':              d['date'],
        'L3_morning':        round(lscore(m, 'L3_atmosphere'), 4),
        'L3_midday':         round(lscore(mid, 'L3_atmosphere'), 4) if mid and lscore(mid, 'L3_atmosphere') is not None else None,
        'L3_evening':        round(lscore(e, 'L3_atmosphere'), 4),
        'L3_night':          round(lscore(ngt, 'L3_atmosphere'), 4) if ngt and lscore(ngt, 'L3_atmosphere') is not None else None,
        'delta_L3':          round(dl3, 4),           # core: morning → evening
        'delta_L3_midday':   dl3_mid,                 # early indicator: morning → midday
        'delta_L3_night':    dl3_ngt,                 # persistence: evening → night
        'delta_L5':          round(dl5, 4),
        'delta_L6':          round(dl6, 4),
        'evening_state':     e['system_state'],
        'evening_L5':        round(lscore(e, 'L5_global_electric_circuit'), 4),
    })

# Search for ΔL3 threshold empirically
anomal_dl3   = [r['delta_L3'] for r in dl3_data if r['evening_state'] == 'anomalous_resonance_state']
seasonal_dl3 = [r['delta_L3'] for r in dl3_data if r['evening_state'] != 'anomalous_resonance_state']

dl3_threshold = None
if anomal_dl3 and seasonal_dl3:
    # lowest anomalous value minus 0.005 as rough threshold
    dl3_threshold = round(min(anomal_dl3) - 0.005, 4)

# ============================================================
# COMBINED ACTIVATION SCORE — better than ΔL3 alone?
# combined = 0.4*ΔL3_norm + 0.3*L5_evening + 0.3*L6_evening
# ============================================================

combined_data = []
for r in dl3_data:
    # normalize ΔL3: typical range ~[-0.2, +0.3] → [0,1]
    dl3_norm = min(1.0, max(0.0, (r['delta_L3'] + 0.2) / 0.5))
    l5_eve   = r['evening_L5']
    l6_eve   = lscore(by_date[r['date']]['evening'], 'L6_resonance_field') or 0

    combined  = round(0.4 * dl3_norm + 0.3 * l5_eve + 0.3 * l6_eve, 4)
    is_anomal = r['evening_state'] == 'anomalous_resonance_state'

    combined_data.append({
        'date':              r['date'],
        'delta_L3':          r['delta_L3'],
        'delta_L3_midday':   r['delta_L3_midday'],
        'delta_L3_night':    r['delta_L3_night'],
        'dl3_norm':          round(dl3_norm, 4),
        'L5_evening':        round(l5_eve, 4),
        'L6_evening':        round(l6_eve, 4),
        'combined':          combined,
        'anomalous':         is_anomal,
    })

anomal_combined   = [d['combined'] for d in combined_data if d['anomalous']]
seasonal_combined = [d['combined'] for d in combined_data if not d['anomalous']]

combined_threshold = None
if anomal_combined and seasonal_combined:
    combined_threshold = round(min(anomal_combined) - 0.01, 4)
    overlap_combined   = max(seasonal_combined) >= min(anomal_combined)
    overlap_dl3        = bool(seasonal_dl3 and anomal_dl3 and
                              max(seasonal_dl3) >= min(anomal_dl3))

print('COMBINED ACTIVATION SCORE')
print('=' * 78)
print(f'  {"Date":<10} {"ΔL3":>7} {"ΔL3mid":>8} {"ΔL3ngt":>8} {"L5eve":>7} {"L6eve":>7} {"combined":>9}  State')
for d in combined_data:
    state   = '⚡ ANOMAL' if d['anomalous'] else '  seasonal'
    mid_str = f'{d["delta_L3_midday"]:>+8.3f}' if d['delta_L3_midday'] is not None else '       —'
    ngt_str = f'{d["delta_L3_night"]:>+8.3f}'  if d['delta_L3_night']  is not None else '       —'
    print(f'  {d["date"]}  {d["delta_L3"]:>+7.3f} {mid_str} {ngt_str} '
          f'{d["L5_evening"]:>7.3f} {d["L6_evening"]:>7.3f} {d["combined"]:>9.4f}  {state}')

if combined_threshold is not None:
    print(f'\n  Combined at anomalous:   mean={np.mean(anomal_combined):.3f}  min={min(anomal_combined):.3f}')
    print(f'  Combined at seasonal:    mean={np.mean(seasonal_combined):.3f}  max={max(seasonal_combined):.3f}')
    print(f'  Threshold combined:      {combined_threshold:.4f}  (Overlap: {overlap_combined})')
    print(f'  Threshold ΔL3 alone:     {dl3_threshold}  (Overlap: {overlap_dl3})')
    better = 'combined better'    if not overlap_combined and overlap_dl3 else \
             'ΔL3 better'         if overlap_combined and not overlap_dl3 else \
             'both equally good'  if not overlap_combined and not overlap_dl3 else \
             'both still uncertain'
    print(f'  → {better}')

COMBINED ACTIVATION SCORE
  Date           ΔL3   ΔL3mid   ΔL3ngt   L5eve   L6eve  combined  State
  2026-05-11   +0.071   +0.047   -0.044   0.364   0.303    0.4168  ⚡ ANOMAL
  2026-05-12   +0.032   +0.017   -0.052   0.479   0.331    0.4286  ⚡ ANOMAL
  2026-05-13   +0.080   -0.009   -0.073   0.394   0.312    0.4359  ⚡ ANOMAL


## 5. Carnegie Amplitude — What drives L5 at evening?

The Carnegie daily cycle is real (all anomalous events fall in the 17–20 UTC window),
but the evening slot (18:30 CEST = 16:30 UTC) is on the rising edge of the
Carnegie curve. The Carnegie peak (~19 UTC) falls between the evening slot (16:30 UTC)
and the night slot (22:30 CEST = 20:30 UTC).
But the **amplitude** varies from day to day. What correlates with a high L5 evening value?

In [6]:
# Correlations with L5_evening across evening snapshots
evening_snaps = [d['evening'] for d in complete_pairs]

if len(evening_snaps) >= 4:
    l5_eve = np.array([lscore(s, 'L5_global_electric_circuit') for s in evening_snaps])

    candidates = {
        'L3_evening':    np.array([lscore(s, 'L3_atmosphere') for s in evening_snaps]),
        'L2_evening':    np.array([lscore(s, 'L2_surface_zone') for s in evening_snaps]),
        'L6_evening':    np.array([lscore(s, 'L6_resonance_field') for s in evening_snaps]),
        'L0_evening':    np.array([lscore(s, 'L0_external_drivers') for s in evening_snaps]),
        'delta_L3':      np.array([d['evening']['layers']['L3_atmosphere']['score']
                                   - d['morning']['layers']['L3_atmosphere']['score']
                                   for d in complete_pairs]),
        'L2_morning':    np.array([lscore(d['morning'], 'L2_surface_zone') for d in complete_pairs]),
    }

    correlations = {}
    for name, arr in candidates.items():
        if arr.std() > 0 and l5_eve.std() > 0:
            r = float(np.corrcoef(l5_eve, arr)[0, 1])
            correlations[name] = round(r, 4)

    correlations_sorted = sorted(correlations.items(), key=lambda x: -abs(x[1]))

    print('CARNEGIE AMPLITUDE — Correlations with L5_evening')
    print('=' * 65)
    print(f'  {"Variable":<20} {"Pearson r":>11}   Interpretation')
    for name, r in correlations_sorted:
        if abs(r) > 0.7:    interp = 'strong correlation'
        elif abs(r) > 0.4:  interp = 'moderate correlation'
        elif abs(r) > 0.2:  interp = 'weak correlation'
        else:               interp = 'no correlation'
        sign = '+' if r >= 0 else '−'
        print(f'  {name:<20} {sign}{abs(r):>10.3f}   {interp}')

    carnegie_amplitude = {
        'n_evenings':      len(evening_snaps),
        'L5_evening_mean': round(float(l5_eve.mean()), 4),
        'L5_evening_std':  round(float(l5_eve.std()),  4),
        'correlations':    correlations,
        'top_predictor':   correlations_sorted[0][0] if correlations_sorted else None,
    }

else:
    print(f'Only {len(evening_snaps)} evening snapshots — correlation not meaningful.')
    print('Activates at >= 4 evening snapshots.')
    carnegie_amplitude = {'n_evenings': len(evening_snaps), 'note': 'insufficient_data'}

Only 3 evening snapshots — correlation not meaningful.
Activates at >= 4 evening snapshots.


## 6. L2→L3 Activation Paradox

L2 (ENSO/SST) remains persistently high, yet L3 (the atmosphere) does not activate accordingly.
We investigate whether L2 and L3 operate on the same timescale, or if L2 acts as a "slow driver."

In [7]:
l2_all  = np.array([lscore(s, 'L2_surface_zone') for s in snaps])
l3_all  = np.array([lscore(s, 'L3_atmosphere')   for s in snaps])
gap_all = l2_all - l3_all

pearson_l2_l3 = float(np.corrcoef(l2_all, l3_all)[0, 1]) if l2_all.std() > 0 else 0.0

# Trend per layer (linear fit over index = time)
def linear_trend(arr):
    if len(arr) < 3 or np.std(arr) == 0:
        return 0.0
    x = np.arange(len(arr))
    slope = float(np.polyfit(x, arr, 1)[0])
    return round(slope, 5)

l2_trend  = linear_trend(l2_all)
l3_trend  = linear_trend(l3_all)
gap_trend = linear_trend(gap_all)

print('L2 ↔ L3 RELATIONSHIP')
print('=' * 65)
print(f'  Pearson L2 vs L3:        {pearson_l2_l3:+.4f}')
print(f'  L2 Trend (Δ/snapshot):   {l2_trend:+.5f}')
print(f'  L3 Trend (Δ/snapshot):   {l3_trend:+.5f}')
print(f'  Gap Trend (Δ/snapshot):  {gap_trend:+.5f}')
print()
print(f'  Gap mean:                {float(gap_all.mean()):.3f}')
print(f'  Gap range:               {float(gap_all.min()):.3f} – {float(gap_all.max()):.3f}')

if pearson_l2_l3 < -0.2:
    interp = 'NEGATIVE correlation: L2 (weekly trend) and L3 (daily rhythm) operate on different timescales.'
elif pearson_l2_l3 > 0.5:
    interp = 'POSITIVE coupling: L2 and L3 move in sync.'
else:
    interp = 'Weak coupling visible — more data needed.'

print(f'\n  → {interp}')

l2_l3_paradox = {
    'pearson':        round(pearson_l2_l3, 4),
    'l2_trend':       l2_trend,
    'l3_trend':       l3_trend,
    'gap_trend':      gap_trend,
    'gap_mean':       round(float(gap_all.mean()), 4),
    'gap_max':        round(float(gap_all.max()),  4),
    'interpretation': interp,
}

L2 ↔ L3 RELATIONSHIP
  Pearson L2 vs L3:        -0.1398
  L2 Trend (Δ/snapshot):   -0.00152
  L3 Trend (Δ/snapshot):   +0.00038
  Gap Trend (Δ/snapshot):  -0.00190

  Gap mean:                0.289
  Gap range:               0.145 – 0.470

  → Weak coupling visible — more data needed.


## 7. Coupling Analysis

Which couplings are stable, and which fluctuate? Variability is often more informative than the mean.

In [8]:
coupling_stats = {}
for s in snaps:
    for c in s.get('couplings', []):
        key = f'{c["from"].split("_")[0]}→{c["to"].split("_")[0]}'
        coupling_stats.setdefault(key, []).append(c['strength'])

coupling_summary = {}
for key, vals in coupling_stats.items():
    coupling_summary[key] = {
        'mean':       round(float(np.mean(vals)), 4),
        'std':        round(float(np.std(vals)),  4),
        'min':        round(float(np.min(vals)),  4),
        'max':        round(float(np.max(vals)),  4),
        'volatility': round(float(np.std(vals) / np.mean(vals)) if np.mean(vals) > 0 else 0, 4),
    }

print('COUPLING PROFILE  (sorted by volatility)')
print('=' * 78)
print(f'  {"Coupling":<10} {"mean":>7} {"std":>7} {"min":>7} {"max":>7} {"CV":>7}  Interpretation')
for key, st in sorted(coupling_summary.items(), key=lambda x: -x[1]['volatility']):
    if st['volatility'] > 0.5:    note = 'highly variable — situation-driven'
    elif st['volatility'] > 0.2:  note = 'moderately variable'
    else:                         note = 'stable — structural'
    print(f'  {key:<10} {st["mean"]:>7.3f} {st["std"]:>7.3f} {st["min"]:>7.3f} {st["max"]:>7.3f} {st["volatility"]:>7.3f}  {note}')

COUPLING PROFILE  (sorted by volatility)
  Coupling      mean     std     min     max      CV  Interpretation
  L5→L6        0.287   0.173   0.121   0.875   0.604  highly variable — situation-driven
  L3→L5        0.257   0.154   0.106   0.778   0.598  highly variable — situation-driven
  L0→L5        0.058   0.012   0.045   0.096   0.214  moderately variable
  L2→L3        0.409   0.076   0.305   0.559   0.185  stable — structural
  L0→L4        0.169   0.023   0.137   0.218   0.135  stable — structural
  L3→L6        0.376   0.038   0.317   0.497   0.101  stable — structural
  L4→L6        0.399   0.039   0.313   0.448   0.097  stable — structural


## 8. Field Operators — Impact Analysis

The Field Operators describe the *driving forces* behind the layers. This section becomes fully active only once multiple snapshots contain operators (Layer 7 was expanded at a later stage).

**What happens here:**
1. Operator statistics across all available snapshots
2. Which operator dominates for a given system state?
3. Correlation: Operator vs. L3 Activation (ΔL3)
4. Ranking of the current operators

In [9]:
# ============================================================
# 8. FIELD OPERATORS — Driving Force Analysis
# ============================================================
# Layer 8 analyzes operators as time series and pattern indicators:
# 8.1 Coverage + regime warning
# 8.2 Statistics + trend (current, previous, delta, rolling)
# 8.3 Lead-lag: Operator(t) → Layer(t+k)
# 8.4 Operator regime (profile classification)
# 8.5 State precursors (operators at t-1)
# 8.6 Operator combinations (simultaneously high)
# 8.7 Operator vs ΔL3 (for hypothesis tracker)
# 8.8 Current ranking
# ============================================================

from itertools import combinations

# Operator names (with backwards compatibility for older snapshots)
OP_NAMES = ['thermal_operator', 'electric_operator', 'ionization_operator',
            'geomagnetic_operator', 'resonance_model_operator',
            'tidal_gravity_operator', 'cross_layer_activation_operator']
OP_ALIASES = {'resonance_operator': 'resonance_model_operator'}

HIGH = 0.5   # threshold "high"
LOW  = 0.3   # threshold "low"

def _get_op_full(snap, op_name):
    """Fetches operator object, handles aliases"""
    ops = snap.get('field_operators') or {}
    o = ops.get(op_name)
    if o is None:
        for old, new in OP_ALIASES.items():
            if new == op_name and old in ops:
                return ops[old]
    return o

def _get_op(snap, op_name):
    """Fetches score only, or None"""
    o = _get_op_full(snap, op_name)
    if o and isinstance(o, dict) and o.get('score') is not None:
        return float(o['score'])
    return None

ops_snaps    = [s for s in snaps if s.get('field_operators')]
ops_count    = len(ops_snaps)
coverage_pct = round(ops_count / len(snaps) * 100, 1) if snaps else 0

# ── 8.1 Coverage + Regime ────────────────────────────────────
if ops_count < 30:
    coverage_regime = 'exploratory'
    coverage_note   = 'exploratory only — operator coverage too low for robust patterns'
elif ops_count < 100:
    coverage_regime = 'first_patterns'
    coverage_note   = 'first patterns visible — interpret with caution'
elif ops_count < 300:
    coverage_regime = 'transition_analysis'
    coverage_note   = 'usable transition analysis possible'
else:
    coverage_regime = 'robust'
    coverage_note   = 'robust seasonal/diurnal analysis possible'

operator_analysis = {
    'snapshots_with_operators': ops_count,
    'total_snapshots':          len(snaps),
    'coverage_pct':             coverage_pct,
    'coverage_regime':          coverage_regime,
    'coverage_note':            coverage_note,
}

print('FIELD OPERATORS')
print('=' * 78)
print(f'  Coverage:  {ops_count}/{len(snaps)} ({coverage_pct}%)')
print(f'  Regime:    {coverage_regime}')
print(f'  → {coverage_note}')

if ops_count == 0:
    print('\n  ❌ No operator data. Layer 7 update required.')
    operator_analysis['status'] = 'no_data'

else:
    # ── 8.2 Statistics + Trend ───────────────────────────────
    op_stats        = {}
    op_trends       = {}
    op_series_cache = {}   # reused in 8.3

    for op in OP_NAMES:
        vals = [_get_op(s, op) for s in ops_snaps]
        op_series_cache[op] = vals
        vals_clean = [v for v in vals if v is not None]

        if not vals_clean:
            op_stats[op]  = None
            op_trends[op] = None
            continue

        op_stats[op] = {
            'count': len(vals_clean),
            'mean':  round(float(np.mean(vals_clean)), 4),
            'std':   round(float(np.std(vals_clean)),  4),
            'min':   round(float(np.min(vals_clean)),  4),
            'max':   round(float(np.max(vals_clean)),  4),
        }

        current  = vals_clean[-1]
        previous = vals_clean[-2] if len(vals_clean) >= 2 else None
        delta    = round(current - previous, 4) if previous is not None else None
        # Rolling: 1d ≈ 4 snapshots, 3d ≈ 12 snapshots
        roll_1d  = round(float(np.mean(vals_clean[-4:])),  4) if len(vals_clean) >= 2 else None
        roll_3d  = round(float(np.mean(vals_clean[-12:])), 4) if len(vals_clean) >= 4 else None

        if delta is None:    direction = 'unknown'
        elif delta > 0.03:   direction = 'rising'
        elif delta < -0.03:  direction = 'falling'
        else:                direction = 'stable'

        op_trends[op] = {
            'current':    round(current, 4),
            'previous':   round(previous, 4) if previous is not None else None,
            'delta':      delta,
            'rolling_1d': roll_1d,
            'rolling_3d': roll_3d,
            'direction':  direction,
        }

    operator_analysis['operator_stats']  = op_stats
    operator_analysis['operator_trends'] = op_trends

    print('\n  ── 8.2 Statistics + Trend ──')
    print(f'  {"Operator":<32} {"curr":>6} {"prev":>6} {"Δ":>7} {"1d":>6} {"3d":>6}  dir')
    for op in OP_NAMES:
        t = op_trends.get(op)
        if not t: continue
        short = op.replace('_operator', '')
        prev  = f'{t["previous"]:.3f}'   if t['previous']   is not None else '   –'
        delt  = f'{t["delta"]:+.3f}'     if t['delta']      is not None else '   –'
        r1    = f'{t["rolling_1d"]:.3f}' if t['rolling_1d'] is not None else '   –'
        r3    = f'{t["rolling_3d"]:.3f}' if t['rolling_3d'] is not None else '   –'
        arrow = {'rising':'↑','falling':'↓','stable':'→','unknown':'?'}[t['direction']]
        print(f'  {short:<32} {t["current"]:>6.3f} {prev:>6} {delt:>7} {r1:>6} {r3:>6}  {arrow} {t["direction"]}')

    # ── 8.3 Lead-Lag Analysis ────────────────────────────────
    LAYERS_TO_PREDICT = ['L3_atmosphere', 'L4_ionosphere',
                         'L5_global_electric_circuit', 'L6_resonance_field']
    LAGS = [1, 2, 4]   # ~6h, ~12h, ~24h at 4 snapshots/day

    lead_lag = {}

    for op in OP_NAMES:
        if not op_stats.get(op): continue
        op_arr = np.array([v if v is not None else np.nan for v in op_series_cache[op]])

        for layer_target in LAYERS_TO_PREDICT:
            tgt = np.array([s['layers'].get(layer_target, {}).get('score')
                            if s['layers'].get(layer_target, {}).get('score') is not None
                            else np.nan for s in ops_snaps])

            for lag in LAGS:
                if len(ops_snaps) <= lag + 3:
                    continue
                a = op_arr[:-lag]
                b = tgt[lag:]
                mask = ~(np.isnan(a) | np.isnan(b))
                if mask.sum() < 4: continue
                a2, b2 = a[mask], b[mask]
                if a2.std() == 0 or b2.std() == 0: continue
                r = float(np.corrcoef(a2, b2)[0, 1])
                if abs(r) > 0.3:
                    key = f'{op.replace("_operator","")}_t__{layer_target}_t+{lag}'
                    lead_lag[key] = {
                        'pearson':       round(r, 4),
                        'n_pairs':       int(mask.sum()),
                        'lag_snapshots': lag,
                    }

    operator_analysis['lead_lag'] = lead_lag

    if lead_lag:
        print('\n  ── 8.3 Lead-Lag: Operator(t) → Layer(t+k)   (|r| > 0.3) ──')
        for key, info in sorted(lead_lag.items(), key=lambda x: -abs(x[1]['pearson'])):
            r        = info['pearson']
            strength = 'strong' if abs(r) > 0.7 else 'moderate' if abs(r) > 0.5 else 'weak'
            print(f'  {key:<52} r = {r:+.3f}  (n={info["n_pairs"]}, {strength})')
    else:
        print(f'\n  ── 8.3 Lead-Lag ──   no correlation with |r| > 0.3 (n_ops={ops_count})')

    # ── 8.4 Operator Regime ──────────────────────────────────
    def classify_regime(snap):
        def val(op): return _get_op(snap, op)
        def hi(op):  v = val(op); return v is not None and v >= HIGH
        def lo(op):  v = val(op); return v is not None and v < LOW

        # Regime 1: Surface Prepared / Atmosphere Delayed
        if hi('cross_layer_activation_operator') and lo('electric_operator'):
            return 'surface_prepared_delayed'
        # Regime 2: Electric Coupling Mode
        if hi('electric_operator') and (val('resonance_model_operator') or 0) >= 0.4:
            return 'electric_coupling_mode'
        # Regime 3: Space Weather Mode
        if hi('ionization_operator') or hi('geomagnetic_operator'):
            return 'space_weather_mode'
        # Regime 4: Quiet Background
        all_low = all((val(op) is None or val(op) < LOW) for op in OP_NAMES)
        if all_low:
            return 'quiet_background'
        return 'mixed'

    regime_assignments = []
    for s in ops_snaps:
        regime_assignments.append({
            'date':   s['_date'],
            'slot':   s['_slot'],
            'state':  s['system_state'],
            'regime': classify_regime(s),
        })
    regime_counts = Counter(r['regime'] for r in regime_assignments)

    operator_analysis['operator_regimes'] = {
        'counts':      dict(regime_counts),
        'assignments': regime_assignments,
    }

    print('\n  ── 8.4 Operator Regime ──')
    for regime, c in regime_counts.most_common():
        pct = round(c / ops_count * 100, 1)
        bar = '█' * c
        print(f'  {regime:<28} {bar} {c}× ({pct}%)')

    # ── 8.5 State Precursors (operators at t-1) ──────────────
    state_precursors = defaultdict(lambda: defaultdict(list))
    for i, s in enumerate(ops_snaps):
        if i == 0: continue
        prev_snap = ops_snaps[i-1]
        cur_state = s['system_state']
        for op in OP_NAMES:
            v = _get_op(prev_snap, op)
            if v is not None:
                state_precursors[cur_state][op].append(v)

    precursor_summary = {}
    for state, ops_dict in state_precursors.items():
        per_op = {op: {'mean_t_minus_1': round(float(np.mean(vals)), 4),
                       'n':              len(vals)}
                  for op, vals in ops_dict.items() if vals}
        top = sorted(per_op.items(), key=lambda x: -x[1]['mean_t_minus_1'])[:3]
        precursor_summary[state] = {
            'all_operators': per_op,
            'top_3':         [{'op': k, **v} for k, v in top],
        }

    operator_analysis['state_precursors'] = precursor_summary

    print('\n  ── 8.5 State Precursors (operators at t-1) ──')
    for state, info in precursor_summary.items():
        top_str = ', '.join(f'{x["op"].replace("_operator","")}={x["mean_t_minus_1"]:.2f}'
                            for x in info['top_3'])
        print(f'  before {state.replace("_state",""):<28} → {top_str}')

    # ── 8.6 Operator Combinations ────────────────────────────
    combo_counts    = Counter()
    combo_followups = defaultdict(list)

    for i, s in enumerate(ops_snaps):
        high_ops = [op.replace('_operator','') for op in OP_NAMES
                    if (_get_op(s, op) or 0) >= HIGH]
        # pairs + triples
        for size in (2, 3):
            for combo in combinations(sorted(high_ops), size):
                combo_counts[combo] += 1
                # track followup for pairs only (more efficient)
                if size == 2 and i + 1 < len(ops_snaps):
                    cur_l3 = s['layers'].get('L3_atmosphere', {}).get('score')
                    nxt_l3 = ops_snaps[i+1]['layers'].get('L3_atmosphere', {}).get('score')
                    if cur_l3 is not None and nxt_l3 is not None:
                        combo_followups[combo].append(nxt_l3 - cur_l3)

    combo_analysis = {}
    for combo, count in combo_counts.most_common(10):
        key   = ' + '.join(combo)
        entry = {'count': count}
        if combo in combo_followups and combo_followups[combo]:
            deltas = combo_followups[combo]
            entry['mean_l3_delta_next'] = round(float(np.mean(deltas)), 4)
            entry['pct_l3_rising_next'] = round(
                sum(1 for d in deltas if d > 0.03) / len(deltas) * 100, 1)
        combo_analysis[key] = entry

    operator_analysis['operator_combinations'] = combo_analysis

    if combo_analysis:
        print('\n  ── 8.6 Operator Combinations (simultaneously high) ──')
        for key, info in list(combo_analysis.items())[:8]:
            extra = ''
            if 'mean_l3_delta_next' in info:
                extra = f'  → ΔL3 next: {info["mean_l3_delta_next"]:+.3f} ({info["pct_l3_rising_next"]:.0f}% rising)'
            print(f'  {key:<48} {info["count"]}×{extra}')

    # ── 8.7 Operator vs ΔL3 (for hypothesis tracker) ─────────
    pairs_with_ops = []
    for d in complete_pairs:
        e = d['evening']
        if not e.get('field_operators'): continue
        dl3 = lscore(e, 'L3_atmosphere') - lscore(d['morning'], 'L3_atmosphere')
        pairs_with_ops.append((dl3, e))

    op_dl3_corr = {}
    if len(pairs_with_ops) >= 4:
        dl3_arr = np.array([p[0] for p in pairs_with_ops])
        for op in OP_NAMES:
            vals = [_get_op(snap, op) for _, snap in pairs_with_ops]
            if all(v is not None for v in vals):
                arr = np.array(vals)
                if arr.std() > 0 and dl3_arr.std() > 0:
                    op_dl3_corr[op] = round(float(np.corrcoef(arr, dl3_arr)[0, 1]), 4)
        if op_dl3_corr:
            print('\n  ── 8.7 Operator vs ΔL3 (activation, evening) ──')
            for op, r in sorted(op_dl3_corr.items(), key=lambda x: -abs(x[1])):
                print(f'  {op.replace("_operator",""):<32} r = {r:+.3f}')
    else:
        print(f'\n  ── 8.7 Operator vs ΔL3 ──   activates at >= 4 day pairs with operators')

    operator_analysis['operator_dl3_correlation'] = op_dl3_corr

    # ── 8.8 Current Ranking ──────────────────────────────────
    latest = ops_snaps[-1]
    ranked = []
    for name in OP_NAMES:
        v = _get_op(latest, name)
        if v is None: continue
        o      = _get_op_full(latest, name)
        interp = o.get('interpretation', '') if isinstance(o, dict) else ''
        ranked.append((name, v, interp))
    ranked.sort(key=lambda x: -x[1])

    print('\n  ── 8.8 Current Operator Ranking ──')
    for name, score, interp in ranked:
        short = name.replace('_operator', '')
        bar   = '▰' * int(score * 10) + '▱' * (10 - int(score * 10))
        print(f'  {short:<32} {bar} {score:.3f}')
        if interp:
            print(f'    └─ {interp[:80]}')

    operator_analysis['latest_ranking'] = [
        {'operator': n, 'score': s, 'interpretation': i} for n, s, i in ranked
    ]
    operator_analysis['status'] = 'active'

FIELD OPERATORS
  Coverage:  4/21 (19.0%)
  Regime:    exploratory
  → exploratory only — operator coverage too low for robust patterns

  ── 8.2 Statistics + Trend ──
  Operator                           curr   prev       Δ     1d     3d  dir
  thermal                           0.290  0.290  +0.001  0.369  0.369  → stable
  electric                          0.317  0.197  +0.119  0.231  0.231  ↑ rising
  ionization                        0.270  0.271  -0.001  0.265  0.265  → stable
  geomagnetic                       0.123  0.107  +0.016  0.104  0.104  → stable
  resonance_model                   0.455  0.425  +0.030  0.432  0.432  → stable
  cross_layer_activation            0.092  0.098  -0.007  0.148  0.148  → stable

  ── 8.3 Lead-Lag ──   no correlation with |r| > 0.3 (n_ops=4)

  ── 8.4 Operator Regime ──
  mixed                        ████ 4× (100.0%)

  ── 8.5 State Precursors (operators at t-1) ──
  before seasonal_transition          → resonance_model=0.42, thermal=0.40, ioni

## 9. State-Transition Matrix

Which state follows which? We consider adjacent snapshots in chronological order.

In [10]:
transitions = []
for i in range(len(snaps) - 1):
    from_state = snaps[i]['system_state']
    to_state   = snaps[i+1]['system_state']
    if from_state != to_state:
        # real transitions only (no 'stay')
        transitions.append({
            'from': from_state,
            'to':   to_state,
            'time': snaps[i+1]['_cest'].strftime('%m-%d %H:%M'),
        })

# Matrix
trans_matrix = defaultdict(lambda: defaultdict(int))
for t in transitions:
    trans_matrix[t['from']][t['to']] += 1

print('STATE TRANSITIONS')
print('=' * 78)
print(f'  Real transitions: {len(transitions)}')
print()
for t in transitions:
    fs = t['from'].replace('_state', '')
    ts = t['to'].replace('_state', '')
    print(f'  {t["time"]:>12}  {fs:<28} → {ts}')

print()
print('TRANSITION COUNTS')
print('=' * 78)
for from_s, to_dict in trans_matrix.items():
    fs = from_s.replace('_state', '')
    for to_s, count in to_dict.items():
        ts = to_s.replace('_state', '')
        print(f'  {fs:<28} → {ts:<28} {count}×')

transition_summary = {
    'total_transitions': len(transitions),
    'transitions':       transitions,
    'matrix':            {k: dict(v) for k, v in trans_matrix.items()},
}

STATE TRANSITIONS
  Real transitions: 11

   05-11 07:16  cavity_condition_shift       → seasonal_transition
   05-11 15:28  seasonal_transition          → cavity_condition_shift
   05-11 20:20  cavity_condition_shift       → anomalous_resonance
   05-11 23:53  anomalous_resonance          → seasonal_transition
   05-12 20:24  seasonal_transition          → anomalous_resonance
   05-12 23:53  anomalous_resonance          → seasonal_transition
   05-13 09:40  seasonal_transition          → cavity_condition_shift
   05-13 20:28  cavity_condition_shift       → anomalous_resonance
   05-13 23:57  anomalous_resonance          → seasonal_transition
   05-14 09:33  seasonal_transition          → cavity_condition_shift
   05-14 12:16  cavity_condition_shift       → seasonal_transition

TRANSITION COUNTS
  cavity_condition_shift       → seasonal_transition          2×
  cavity_condition_shift       → anomalous_resonance          2×
  seasonal_transition          → cavity_condition_shift       3

In [11]:
# ============================================================
# EVIDENCE ASSESSMENT — scientifically cautious
# ============================================================

def evidence_level(n):
    """Classifies sample size into evidence level"""
    if n < 10:  return 'exploratory_signal'
    if n < 30:  return 'testable'
    if n < 100: return 'moderate_evidence'
    return 'robust_candidate'

def make_hypothesis(id, question, status, evidence_str, n, next_step):
    """Creates hypothesis with automatic evidence assessment"""
    ev_level = evidence_level(n)
    # Downgrade status if n is too small
    corrected_status = status
    if ev_level == 'exploratory_signal' and status in ('confirmed', 'likely'):
        corrected_status = 'open'
    elif ev_level == 'testable' and status == 'confirmed':
        corrected_status = 'likely'
    return {
        'id':              id,
        'question':        question,
        'status':          corrected_status,
        'status_original': status,   # before correction
        'evidence':        evidence_str,
        'n':               n,
        'evidence_level':  ev_level,
        'next_step':       next_step,
    }

## 10. Hypothesis Tracker

Hypotheses are generated **based on rules**: Condition met → Hypothesis active, with current evidence status.

In [12]:
hypotheses = []

# H1: Carnegie Cycle ─────────────────────────────────────────
n_evenings_total = sum(1 for s in snaps if s['_slot'] == 'evening')
n_anomal_evening = sum(1 for s in snaps
                       if s['_slot'] == 'evening' and s['system_state'] == 'anomalous_resonance_state')
n_anomal_morning = sum(1 for s in snaps
                       if s['_slot'] == 'morning' and s['system_state'] == 'anomalous_resonance_state')
n_anomal_total   = sum(1 for s in snaps if s['system_state'] == 'anomalous_resonance_state')

if n_anomal_total > 0:
    pct_in_evening = round(n_anomal_evening / n_anomal_total * 100, 1)
    status = 'confirmed' if pct_in_evening >= 90 else 'likely' if pct_in_evening >= 70 else 'mixed'
    hypotheses.append(make_hypothesis(
        id           = 'H1',
        question     = 'Does anomalous_resonance_state occur exclusively in the evening slot (>= 18 CEST)?',
        status       = status,
        evidence_str = f'{n_anomal_evening}/{n_anomal_total} anomalous events in evening slot ({pct_in_evening}%). Morning: {n_anomal_morning}.',
        n            = n_anomal_total,
        next_step    = 'More snapshots; add midday slot for confirmation if needed.',
    ))

# H2: ΔL3 threshold ──────────────────────────────────────────
if anomal_dl3 and seasonal_dl3:
    overlap = max(seasonal_dl3) >= min(anomal_dl3)
    status  = 'likely' if not overlap else 'mixed'

# H3: L2-L3 Paradox ──────────────────────────────────────────
if pearson_l2_l3 < -0.2:
    hypotheses.append({
        'id':       'H3',
        'question': 'Does an L2→L3 activation paradox exist (rising L2, falling L3)?',
        'status':   'open',
        'evidence': f'Pearson L2 vs L3 = {pearson_l2_l3:+.3f}. L2 trend {l2_trend:+.5f} (rising). L3 trend {l3_trend:+.5f}. Gap growing by {gap_trend:+.5f}/snapshot.',
        'next_step': 'Check lagged correlation L2(t) vs L3(t+24h, t+48h, t+7d). Possible: saturation effect or missing trigger (wind shear, cold air intrusion) in L3 model.',
    })

# H4: Carnegie Amplitude Modulator ───────────────────────────
if 'correlations' in carnegie_amplitude:
    top_var, top_r = max(carnegie_amplitude['correlations'].items(), key=lambda x: abs(x[1]))
    if abs(top_r) > 0.5:
        hypotheses.append(make_hypothesis(
            id           = 'H4',
            question     = f'Does {top_var} modulate the Carnegie amplitude (L5 evening)?',
            status       = 'likely' if abs(top_r) > 0.7 else 'open',
            evidence_str = f'Pearson L5_evening vs {top_var} = {top_r:+.3f} over {carnegie_amplitude["n_evenings"]} evenings.',
            n            = carnegie_amplitude['n_evenings'],
            next_step    = 'Confirm correlation in larger sample.',
        ))

# H_combined: Combined Score vs ΔL3 alone ────────────────────
if combined_threshold is not None and anomal_combined:
    hypotheses.append(make_hypothesis(
        id           = 'H_combined',
        question     = 'Is combined_activation_score (ΔL3+L5+L6) better than ΔL3 alone?',
        status       = 'likely' if not overlap_combined and overlap_dl3 else 'open',
        evidence_str = (f'combined overlap={overlap_combined} vs ΔL3 overlap={overlap_dl3}. '
                        f'combined threshold={combined_threshold:.4f}.'),
        n            = len(complete_pairs),
        next_step    = 'More day pairs for robust separation. ROC analysis from n>=20.',
    ))

# H5: Operator-based hypotheses ──────────────────────────────
if operator_analysis.get('status') == 'active':
    if operator_analysis.get('operator_dl3_correlation'):
        for op, r in operator_analysis['operator_dl3_correlation'].items():
            if abs(r) > 0.5:
                short = op.replace('_operator', '')
                hypotheses.append({
                    'id':       f'H_op_{short}',
                    'question': f'Is the {short} operator a predictor for ΔL3 activation?',
                    'status':   'likely' if abs(r) > 0.7 else 'open',
                    'evidence': f'Pearson {op} vs ΔL3 = {r:+.3f}.',
                    'next_step': 'Confirm with more operator snapshots.',
                })

# H6: Persistent background tags ─────────────────────────────
all_tags = []
for s in snaps:
    all_tags.extend(s.get('event_tags', []))
tag_freq = Counter(all_tags)
persistent_tags = [t for t, c in tag_freq.items()
                   if c == len(snaps) and not t.startswith('state_')]
if persistent_tags:
    hypotheses.append({
        'id':       'H6',
        'question': 'Which tags are persistent background vs actual activation signal?',
        'status':   'open',
        'evidence': f'Present in 100% of all snapshots: {", ".join(persistent_tags)}. These are seasonal background, not activation signals.',
        'next_step': 'Separate background tags from Layer 7 tag list or mark as baseline_tags.',
    })

# H7: Lead-lag operators ─────────────────────────────────────
if operator_analysis.get('lead_lag'):
    for key, info in operator_analysis['lead_lag'].items():
        if abs(info['pearson']) > 0.6:
            hypotheses.append({
                'id':       f'H_leadlag_{key}',
                'question': f'Does {key.split("__")[0]} predict the value of {key.split("__")[1]}?',
                'status':   'likely' if abs(info['pearson']) > 0.7 else 'open',
                'evidence': f'Pearson = {info["pearson"]:+.3f} (n={info["n_pairs"]}, lag {info["lag_snapshots"]} snapshots).',
                'next_step': 'Confirm with more snapshots; define prediction threshold.',
            })

# H8: Dominant operator regime ───────────────────────────────
if operator_analysis.get('operator_regimes'):
    rc = operator_analysis['operator_regimes']['counts']
    if rc:
        top_regime, top_count = max(rc.items(), key=lambda x: x[1])
        pct = round(top_count / ops_count * 100, 1)
        if pct >= 40:
            hypotheses.append({
                'id':       'H8',
                'question': f'Is "{top_regime}" the dominant operator regime?',
                'status':   'likely' if pct >= 60 else 'open',
                'evidence': f'{top_count}/{ops_count} snapshots ({pct}%) in regime "{top_regime}".',
                'next_step': 'Analyze transition patterns between regimes (Layer 9?).',
            })

# Output
icon    = {'confirmed':'✅', 'likely':'🔶', 'open':'❓', 'mixed':'⚠️'}
ev_icon = {'exploratory_signal':'🔬', 'testable':'🧪', 'moderate_evidence':'📊', 'robust_candidate':'✔️'}
print('HYPOTHESIS TRACKER')
print('=' * 78)
for h in hypotheses:
    ic        = icon.get(h['status'], '?')
    ev        = ev_icon.get(h.get('evidence_level', ''), '')
    corrected = ' (corrected)' if h.get('status') != h.get('status_original') else ''
    print(f'\n  {ic} {h["id"]}: {h["question"]}')
    print(f'     Status:    {h["status"]}{corrected}  {ev} {h.get("evidence_level","")}  (n={h.get("n","?")})')
    print(f'     Evidence:  {h["evidence"]}')
    print(f'     → {h["next_step"]}')

HYPOTHESIS TRACKER

  ❓ H1: Does anomalous_resonance_state occur exclusively in the evening slot (>= 18 CEST)?
     Status:    open (corrected)  🔬 exploratory_signal  (n=3)
     Evidence:  3/3 anomalous events in evening slot (100.0%). Morning: 0.
     → More snapshots; add midday slot for confirmation if needed.

  ❓ H6: Which tags are persistent background vs actual activation signal?
     Status:    open (corrected)     (n=?)
     Evidence:  Present in 100% of all snapshots: el_nino_developing, non_geometric_dominance. These are seasonal background, not activation signals.
     → Separate background tags from Layer 7 tag list or mark as baseline_tags.

  🔶 H8: Is "mixed" the dominant operator regime?
     Status:    likely (corrected)     (n=?)
     Evidence:  4/4 snapshots (100.0%) in regime "mixed".
     → Analyze transition patterns between regimes (Layer 9?).


## 11. Visualizations

In [13]:
# Plot 1: ΔL3 per day with state marker
if dl3_data:
    dates_   = [r['date'] for r in dl3_data]
    dl3_vals = [r['delta_L3'] for r in dl3_data]
    colors   = ['#e74c3c' if r['evening_state'] == 'anomalous_resonance_state' else '#3498db'
                for r in dl3_data]

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        x=dates_, y=dl3_vals, marker_color=colors,
        text=[f'{v:+.3f}' for v in dl3_vals], textposition='outside',
        textfont=dict(color='white'),
    ))

    # ΔL3 midday markers (early activation indicator)
    mid_dates = [r['date'] for r in dl3_data if r.get('delta_L3_midday') is not None]
    mid_vals  = [r['delta_L3_midday'] for r in dl3_data if r.get('delta_L3_midday') is not None]
    if mid_dates:
        fig1.add_trace(go.Scatter(
            x=mid_dates, y=mid_vals,
            mode='markers',
            marker=dict(symbol='diamond', size=9, color='#f39c12'),
            name='ΔL3 midday (06:30→12:30)',
        ))

    if dl3_threshold is not None:
        fig1.add_hline(y=dl3_threshold, line_dash='dash', line_color='#f39c12',
                       annotation_text=f'Threshold ΔL3 = {dl3_threshold:+.3f}',
                       annotation_position='top right',
                       annotation_font_color='#f39c12')

    fig1.update_layout(
        title=dict(text='ΔL3 (L3 evening 18:30 − L3 morning 06:30) per day',
                   font=dict(size=14, color='white')),
        xaxis=dict(title='Date', color='white', gridcolor='#222244'),
        yaxis=dict(title='ΔL3', color='white', gridcolor='#222244'),
        height=380, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
        showlegend=True,
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='white', size=10)),
        margin=dict(l=60, r=40, t=55, b=50),
    )
    fig1.show()

In [14]:
# ============================================================
# Plot 2: Field Operators — Multi-Panel
# Panel A: Current values + trend indicator
# Panel B: Operator time series
# Panel C: Operator regime frequency
# ============================================================

if operator_analysis.get('status') == 'active' and operator_analysis.get('latest_ranking'):
    ranking     = operator_analysis['latest_ranking']
    trends_data = operator_analysis.get('operator_trends', {})

    fig2 = make_subplots(
        rows=2, cols=2,
        specs=[[{'colspan': 2}, None],
               [{}, {}]],
        subplot_titles=(
            'Current Operators with Trend',
            'Operator Time Series',
            'Operator Regime Frequency',
        ),
        row_heights=[0.42, 0.58],
        vertical_spacing=0.18,
        horizontal_spacing=0.12,
    )

    # ── Panel A: Current values with trend arrows ─────────────
    names_short = [r['operator'].replace('_operator', '').replace('_', ' ') for r in ranking]
    scores      = [r['score'] for r in ranking]
    colors      = ['#2ecc71' if s < 0.3 else '#f39c12' if s < 0.6 else '#e74c3c' for s in scores]

    # Trend arrows + delta text from operator_trends
    arrow_map   = {'rising': '↑', 'falling': '↓', 'stable': '→', 'unknown': '?'}
    text_labels = []
    for r in ranking:
        t = trends_data.get(r['operator'])
        if t and t.get('delta') is not None:
            arrow = arrow_map.get(t.get('direction', 'unknown'), '?')
            text_labels.append(f'{r["score"]:.3f}  {arrow} Δ{t["delta"]:+.3f}')
        else:
            text_labels.append(f'{r["score"]:.3f}')

    fig2.add_trace(go.Bar(
        x=scores, y=names_short, orientation='h',
        marker_color=colors, opacity=0.85,
        text=text_labels, textposition='outside',
        textfont=dict(color='white', size=11),
        showlegend=False,
    ), row=1, col=1)
    fig2.add_vline(x=0.3, line_dash='dot', line_color='#888780', row=1, col=1)
    fig2.add_vline(x=0.6, line_dash='dot', line_color='#f39c12', row=1, col=1)
    fig2.update_xaxes(range=[0, 1.25], gridcolor='#222244', row=1, col=1)

    # ── Panel B: Operator time series ────────────────────────
    op_palette = {
        'thermal_operator':                '#e74c3c',
        'electric_operator':               '#f39c12',
        'ionization_operator':             '#9b59b6',
        'geomagnetic_operator':            '#3498db',
        'resonance_model_operator':        '#1abc9c',
        'tidal_gravity_operator':          '#95a5a6',
        'cross_layer_activation_operator': '#e67e22',
    }

    ops_timestamps = [s['_cest'] for s in ops_snaps]

    for op in OP_NAMES:
        vals = [_get_op(s, op) for s in ops_snaps]
        if all(v is None for v in vals):
            continue
        y_plot = [v if v is not None else None for v in vals]
        fig2.add_trace(go.Scatter(
            x=ops_timestamps, y=y_plot,
            name=op.replace('_operator', ''),
            mode='lines+markers',
            line=dict(color=op_palette.get(op, '#ffffff'), width=2),
            marker=dict(size=6),
            connectgaps=False,
        ), row=2, col=1)
    fig2.add_hline(y=HIGH, line_dash='dot', line_color='#e74c3c',
                   line_width=1, row=2, col=1)
    fig2.add_hline(y=LOW,  line_dash='dot', line_color='#888780',
                   line_width=1, row=2, col=1)
    fig2.update_xaxes(gridcolor='#222244', row=2, col=1)
    fig2.update_yaxes(title_text='Score', range=[0, 1.05],
                      gridcolor='#222244', row=2, col=1)

    # ── Panel C: Operator regime frequency ───────────────────
    regime_counts = operator_analysis.get('operator_regimes', {}).get('counts', {})
    if regime_counts:
        regime_palette = {
            'surface_prepared_delayed': '#e67e22',
            'electric_coupling_mode':   '#f39c12',
            'space_weather_mode':       '#9b59b6',
            'quiet_background':         '#2ecc71',
            'mixed':                    '#888780',
        }
        sorted_regimes = sorted(regime_counts.items(), key=lambda x: -x[1])
        reg_names  = [r[0] for r in sorted_regimes]
        reg_counts = [r[1] for r in sorted_regimes]
        reg_colors = [regime_palette.get(r, '#666666') for r in reg_names]

        fig2.add_trace(go.Bar(
            x=reg_counts, y=reg_names, orientation='h',
            marker_color=reg_colors, opacity=0.85,
            text=[f'{c}× ({round(c/ops_count*100)}%)' for c in reg_counts],
            textposition='outside', textfont=dict(color='white', size=10),
            showlegend=False,
        ), row=2, col=2)
        fig2.update_xaxes(gridcolor='#222244', row=2, col=2,
                          range=[0, max(reg_counts) * 1.4])
    else:
        fig2.add_annotation(
            text='no regime data', x=0.5, y=0.5,
            xref='x4', yref='y4', showarrow=False,
            font=dict(color='#888', size=11),
            row=2, col=2,
        )

    # ── Layout ───────────────────────────────────────────────
    fig2.update_layout(
        height=700,
        plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
        font=dict(color='white'),
        margin=dict(l=180, r=60, t=70, b=40),
        legend=dict(
            orientation='h', y=-0.08, x=0.5, xanchor='center',
            bgcolor='rgba(0,0,0,0)', font=dict(color='white', size=10),
        ),
        title=dict(
            text=f'Field Operators — Coverage {coverage_pct}% ({coverage_regime})',
            font=dict(size=14, color='white'),
            x=0.5, xanchor='center',
        ),
    )
    for r, c in [(1,1),(2,1),(2,2)]:
        fig2.update_yaxes(color='white', tickfont=dict(size=10), row=r, col=c)
        fig2.update_xaxes(color='white', tickfont=dict(size=10), row=r, col=c)

    fig2.show()

In [15]:
# Plot 3: System-State Timeline + L5 time series
ts_x   = [s['_cest'] for s in snaps]
l5_y   = [lscore(s, 'L5_global_electric_circuit') for s in snaps]
l3_y   = [lscore(s, 'L3_atmosphere') for s in snaps]
states = [s['system_state'] for s in snaps]

state_colors = {
    'seasonal_transition_state':    '#3498db',
    'anomalous_resonance_state':    '#e74c3c',
    'cavity_condition_shift_state': '#f39c12',
    'normal_background_state':      '#2ecc71',
}

fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                     subplot_titles=('L5 (GEC) and L3 (Atmosphere)', 'System State'))

fig3.add_trace(go.Scatter(x=ts_x, y=l5_y, name='L5', mode='lines+markers',
                          line=dict(color='#e74c3c'), marker=dict(size=8)), row=1, col=1)
fig3.add_trace(go.Scatter(x=ts_x, y=l3_y, name='L3', mode='lines+markers',
                          line=dict(color='#3498db'), marker=dict(size=8)), row=1, col=1)

state_y = [list(state_colors.keys()).index(st) if st in state_colors else 0 for st in states]
fig3.add_trace(go.Scatter(
    x=ts_x, y=state_y, mode='markers', showlegend=False,
    marker=dict(size=12, color=[state_colors.get(st, '#888') for st in states]),
    text=states, hovertemplate='%{text}<extra></extra>',
), row=2, col=1)

fig3.update_layout(
    height=520, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='white')),
    margin=dict(l=60, r=40, t=60, b=40),
)
fig3.update_xaxes(gridcolor='#222244', color='white')
fig3.update_yaxes(gridcolor='#222244', color='white')
fig3.update_yaxes(tickvals=list(range(len(state_colors))),
                  ticktext=[s.replace('_state', '') for s in state_colors.keys()],
                  row=2, col=1)
fig3.show()

## 12. Export

`layer8_test_state.json` — machine-readable, for Layer 9 or subsequent analysis.
`layer8_test_report.md` — human-readable report.

In [16]:
layer8_state = {
    'timestamp':           RUN_TS,
    'engine_version':      ENGINE_VERSION,
    'layer':               8,
    'name':                'Research & Hypothesis Engine',
    'snapshots_analyzed':  len(snaps),
    'day_pairs_complete':  n_pairs,           # morning + evening
    'day_quads_complete':  len(complete_full), # all 4 slots

    'state_frequency':     state_freq,
    'layer_stats':         layer_stats,
    'dominance_counts':    dict(dom_counts),

    'morning_evening_analysis': {
        'day_pairs':          dl3_data,
        'dl3_threshold':      dl3_threshold,
        'dl3_anomal_min':     round(min(anomal_dl3), 4) if anomal_dl3 else None,
        'dl3_seasonal_max':   round(max(seasonal_dl3), 4) if seasonal_dl3 else None,
        'midday_available':   sum(1 for r in dl3_data if r.get('delta_L3_midday') is not None),
        'night_available':    sum(1 for r in dl3_data if r.get('delta_L3_night')  is not None),
    },

    'carnegie_amplitude':  carnegie_amplitude,
    'l2_l3_paradox':       l2_l3_paradox,
    'coupling_summary':    coupling_summary,
    'operator_analysis':   operator_analysis,
    'transitions':         transition_summary,

    'hypotheses':          hypotheses,

    'top_findings': [
        h['question'] for h in hypotheses if h['status'] in ('confirmed', 'likely')
    ],
}

with open(STATE_FILE, 'w', encoding='utf-8') as f:
    json.dump(layer8_state, f, indent=2, ensure_ascii=False, default=str)

print(f'✅ {STATE_FILE} saved ({len(json.dumps(layer8_state, default=str))} bytes)')


✅ ../data/states/layer8_test_state.json saved (9390 bytes)


In [17]:
# Markdown report
lines = []
lines.append(f'# Layer 8 — Research Report')
lines.append(f'')
lines.append(f'**Run:** {RUN_TS}')
lines.append(f'**Snapshots analyzed:** {len(snaps)}')
lines.append(f'**Complete day pairs:** {n_pairs}  (morning + evening)')
lines.append(f'**Complete day quads:** {len(complete_full)}  (all 4 slots)')
lines.append(f'')
lines.append(f'## System State Frequency')
lines.append(f'')
for state, st in state_freq.items():
    lines.append(f'- `{state}` — {st["count"]}× ({st["pct"]}%)')
lines.append(f'')
lines.append(f'## Day Pairs (ΔL3 Activation)')
lines.append(f'')
lines.append(f'| Date | L3 morning | L3 midday | L3 evening | ΔL3 | ΔL3 midday | Evening State |')
lines.append(f'|---|---|---|---|---|---|---|')
for r in dl3_data:
    state_short = r['evening_state'].replace('_state', '')
    mid_str = f'{r["delta_L3_midday"]:+.3f}' if r.get('delta_L3_midday') is not None else '—'
    l3_mid  = f'{r["L3_midday"]:.3f}'         if r.get('L3_midday')       is not None else '—'
    lines.append(f'| {r["date"]} | {r["L3_morning"]:.3f} | {l3_mid} | {r["L3_evening"]:.3f} | {r["delta_L3"]:+.3f} | {mid_str} | {state_short} |')
if dl3_threshold is not None:
    lines.append(f'')
    lines.append(f'**ΔL3 threshold (empirical):** {dl3_threshold:+.3f}')
lines.append(f'')
lines.append(f'## L2 ↔ L3 Relationship')
lines.append(f'')
lines.append(f'- Pearson: **{pearson_l2_l3:+.3f}**')
lines.append(f'- L2 trend: {l2_trend:+.5f} / snapshot')
lines.append(f'- L3 trend: {l3_trend:+.5f} / snapshot')
lines.append(f'- Gap trend: {gap_trend:+.5f} / snapshot')
lines.append(f'- {l2_l3_paradox["interpretation"]}')
lines.append(f'')
lines.append(f'## Field Operators')
lines.append(f'')
if operator_analysis.get('status') == 'active':
    lines.append(f'Coverage: {operator_analysis["snapshots_with_operators"]}/{operator_analysis["total_snapshots"]} snapshots')
    lines.append(f'')
    if operator_analysis.get('latest_ranking'):
        lines.append(f'### Current Operator Ranking')
        lines.append(f'')
        for r in operator_analysis['latest_ranking']:
            short = r['operator'].replace('_operator', '')
            lines.append(f'- **{short}**: {r["score"]:.3f} — {r["interpretation"]}')
    if operator_analysis.get('operator_dl3_correlation'):
        lines.append(f'')
        lines.append(f'### Operator ↔ ΔL3 Correlation')
        lines.append(f'')
        for op, r in sorted(operator_analysis['operator_dl3_correlation'].items(),
                             key=lambda x: -abs(x[1])):
            short = op.replace('_operator', '')
            lines.append(f'- {short}: r = {r:+.3f}')
else:
    lines.append(f'_{operator_analysis.get("status", "unknown")}_ — more snapshots needed.')
lines.append(f'')
lines.append(f'## Hypotheses')
lines.append(f'')
status_emoji = {'confirmed': '✅', 'likely': '🔶', 'open': '❓', 'mixed': '⚠️'}
for h in hypotheses:
    em = status_emoji.get(h['status'], '•')
    lines.append(f'### {em} {h["id"]}: {h["question"]}')
    lines.append(f'')
    lines.append(f'- **Status:** {h["status"]}')
    lines.append(f'- **Evidence:** {h["evidence"]}')
    lines.append(f'- **Next step:** {h["next_step"]}')
    lines.append(f'')

with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'✅ {REPORT_FILE} saved')

✅ ../research/layer8_test_report.md saved


In [18]:
# ============================================================
# 13. CAVITY EVENT STUDY
# Analyzes every cavity_condition_shift_state in context
# ============================================================

cavity_events = []
for i, s in enumerate(snaps):
    if s['system_state'] != 'cavity_condition_shift_state':
        continue

    t_minus = snaps[i-1] if i > 0 else None
    t_zero  = s
    t_plus  = snaps[i+1] if i < len(snaps)-1 else None

    def snap_layer(snap, lname):
        if snap is None: return None
        return snap['layers'].get(lname, {}).get('score')

    def snap_op(snap, op):
        if snap is None: return None
        ops = snap.get('field_operators') or {}
        o = ops.get(op) or ops.get('resonance_operator')
        if isinstance(o, dict): return o.get('score')
        return None

    def snap_coupling(snap, from_l, to_l):
        if snap is None: return None
        for c in snap.get('couplings', []):
            if c['from'] == from_l and c['to'] == to_l:
                return c['strength']
        return None

    # Layer trajectories
    layers_track = {}
    for lname in ['L3_atmosphere', 'L4_ionosphere',
                  'L5_global_electric_circuit', 'L6_resonance_field']:
        layers_track[lname] = {
            't-1': snap_layer(t_minus, lname),
            't':   snap_layer(t_zero,  lname),
            't+1': snap_layer(t_plus,  lname),
        }

    # Couplings at t
    l4_l6 = snap_coupling(t_zero, 'L4_ionosphere', 'L6_resonance_field')
    l5_l6 = snap_coupling(t_zero, 'L5_global_electric_circuit', 'L6_resonance_field')

    # Operators at t
    res_op   = snap_op(t_zero, 'resonance_model_operator')
    cross_op = snap_op(t_zero, 'cross_layer_activation_operator')

    # Downstream confirmation: L5+L6 after cavity event
    l5_after = snap_layer(t_plus, 'L5_global_electric_circuit')
    l6_after = snap_layer(t_plus, 'L6_resonance_field')

    # Classification
    if t_plus is not None:
        next_state = t_plus['system_state']
        if next_state == 'anomalous_resonance_state':
            outcome = 'cavity_shift_precursor'
        elif next_state == 'cavity_condition_shift_state':
            outcome = 'cavity_shift_persistent'
        elif (l5_after or 0) < 0.3 and (l6_after or 0) < 0.3:
            outcome = 'cavity_shift_failed'
        else:
            outcome = 'cavity_shift_neutral'
    else:
        outcome = 'unknown_no_followup'

    event = {
        'timestamp':     t_zero['_cest'].strftime('%Y-%m-%d %H:%M'),
        'slot':          t_zero['_slot'],
        'outcome':       outcome,
        'state_t_minus': t_minus['system_state'].replace('_state', '') if t_minus else '—',
        'state_t':       'cavity_condition_shift',
        'state_t_plus':  t_plus['system_state'].replace('_state', '')  if t_plus  else '—',
        'layers':        layers_track,
        'couplings': {
            'L4_to_L6': round(l4_l6, 3) if l4_l6 is not None else None,
            'L5_to_L6': round(l5_l6, 3) if l5_l6 is not None else None,
        },
        'operators': {
            'resonance_model':        round(res_op,   3) if res_op   is not None else None,
            'cross_layer_activation': round(cross_op, 3) if cross_op is not None else None,
        },
    }
    cavity_events.append(event)

outcome_counts = Counter(e['outcome'] for e in cavity_events)

print('CAVITY EVENT STUDY')
print('=' * 78)
print(f'  Cavity events total: {len(cavity_events)}')
print()
for e in cavity_events:
    print(f'  ── {e["timestamp"]} CEST ({e["slot"]}) ──')
    print(f'     {e["state_t_minus"]} → cavity_shift → {e["state_t_plus"]}')
    print(f'     Outcome: {e["outcome"]}')
    for lname, vals in e['layers'].items():
        short = lname.split('_')[0]
        tm  = f'{vals["t-1"]:.3f}' if vals['t-1'] is not None else '  —  '
        t0  = f'{vals["t"]:.3f}'   if vals['t']   is not None else '  —  '
        tp  = f'{vals["t+1"]:.3f}' if vals['t+1'] is not None else '  —  '
        print(f'     {short:<4}  t-1={tm}  t={t0}  t+1={tp}')
    print(f'     L4→L6={e["couplings"]["L4_to_L6"]}  '
          f'L5→L6={e["couplings"]["L5_to_L6"]}  '
          f'res_op={e["operators"]["resonance_model"]}  '
          f'cross_op={e["operators"]["cross_layer_activation"]}')
    print()

print('Outcome distribution:')
for outcome, c in outcome_counts.most_common():
    print(f'  {outcome:<30} {c}×')

CAVITY EVENT STUDY
  Cavity events total: 7

  ── 2026-05-10 11:01 CEST (midday) ──
     — → cavity_shift → cavity_condition_shift
     Outcome: cavity_shift_persistent
     L3    t-1=  —    t=0.172  t+1=0.139
     L4    t-1=  —    t=0.333  t+1=0.344
     L5    t-1=  —    t=0.227  t+1=0.268
     L6    t-1=  —    t=0.283  t+1=0.267
     L4→L6=0.428  L5→L6=0.121  res_op=None  cross_op=None

  ── 2026-05-10 13:41 CEST (midday) ──
     cavity_condition_shift → cavity_shift → cavity_condition_shift
     Outcome: cavity_shift_persistent
     L3    t-1=0.172  t=0.139  t+1=0.168
     L4    t-1=0.333  t=0.344  t+1=0.336
     L5    t-1=0.227  t=0.268  t+1=0.248
     L6    t-1=0.283  t=0.267  t+1=0.288
     L4→L6=0.428  L5→L6=0.253  res_op=None  cross_op=None

  ── 2026-05-10 19:31 CEST (evening) ──
     cavity_condition_shift → cavity_shift → seasonal_transition
     Outcome: cavity_shift_failed
     L3    t-1=0.139  t=0.168  t+1=0.108
     L4    t-1=0.344  t=0.336  t+1=0.292
     L5    t-1=0.26

In [19]:
# ============================================================
# 14. HYPOTHESIS FACTORY (v2)
# Scientifically cautious: status ≠ evidence
# ============================================================

import os

CANDIDATES_DIR = '../research/test_hypothesis_candidates'
REGISTRY_FILE  = '../research/test_hypothesis_registry.json'
os.makedirs(CANDIDATES_DIR, exist_ok=True)

# ── Helper functions ─────────────────────────────────────────

def evidence_level(n):
    if n is None:     return 'unknown'
    if n < 10:        return 'exploratory_signal'
    if n < 30:        return 'testable'
    if n < 100:       return 'moderate_evidence'
    return 'robust_candidate'

def promotion_blocked_reason(n, pattern_status, review_status):
    """Explains why a hypothesis is not yet eligible for model promotion"""
    if review_status != 'reviewed':
        return 'not_manually_reviewed'
    if n is None or n < 10:
        return 'insufficient_sample'
    if pattern_status == 'absent':
        return 'pattern_not_detected'
    return None

def make_candidate(
    id, type, question,
    pattern_status,       # detected / absent / unclear
    n, source_metric,
    evidence_str,
    next_step,
    effect_size=None,
    pearson=None,
    lag=None,
    mechanism=None,
    priority='medium',    # high / medium / low / quarantine
    category='core',      # core / diagnostic / exploratory_leadlag / waiting
):
    ev_level = evidence_level(n)

    # Missing evidence → immediately flagged as candidate without status
    if n is None or ev_level == 'unknown':
        pat_status = 'unclear'
        blocked    = 'missing_evidence_metadata'
    else:
        pat_status = pattern_status
        blocked    = promotion_blocked_reason(n, pat_status, 'unreviewed')

    return {
        'id':                       id,
        'type':                     type,
        'question':                 question,
        'pattern_status':           pat_status,
        'evidence_level':           ev_level,
        'n':                        n,
        'effect_size':              effect_size,
        'pearson':                  pearson,
        'lag':                      lag,
        'source_metric':            source_metric,
        'mechanism':                mechanism,
        'evidence':                 evidence_str,
        'next_step':                next_step,
        'priority':                 priority,
        'category':                 category,
        'review_status':            'unreviewed',
        'include_in_report':        False,
        'include_in_model_logic':   False,
        'promotion_blocked_reason': blocked,
        'generated_at':             RUN_TS,
        'source_snapshots':         len(snaps),
    }

# ── Define candidates ────────────────────────────────────────

factory_candidates = []

# ── CORE CANDIDATES ──────────────────────────────────────────

# H_combined — highest priority
if combined_threshold is not None and anomal_combined:
    factory_candidates.append(make_candidate(
        id             = 'H_combined',
        type           = 'predictive_candidate',
        question       = 'Does combined_activation_score (ΔL3+L5+L6) predict anomalous_resonance better than ΔL3 alone?',
        pattern_status = 'detected' if not overlap_combined else 'unclear',
        n              = len(complete_pairs),
        source_metric  = 'combined_activation_score',
        effect_size    = round(min(anomal_combined) - max(seasonal_combined), 4) if not overlap_combined else 0.0,
        evidence_str   = (f'combined overlap={overlap_combined} (threshold={combined_threshold:.4f}). '
                          f'ΔL3 overlap={overlap_dl3}. '
                          f'anomalous mean={np.mean(anomal_combined):.3f}, '
                          f'seasonal mean={np.mean(seasonal_combined):.3f}.'),
        mechanism      = 'ΔL3 measures atmospheric activation, L5/L6 provide downstream confirmation — combined yields a more robust signal',
        next_step      = 'ROC curve from n>=20 day pairs. Formally test threshold.',
        priority       = 'high',
        category       = 'core',
    ))

# H1 — Carnegie diurnal pattern
if n_anomal_total > 0:
    pct = round(n_anomal_evening / n_anomal_total * 100, 1)
    factory_candidates.append(make_candidate(
        id             = 'H1',
        type           = 'temporal_pattern',
        question       = 'Does anomalous_resonance_state occur exclusively in the evening slot (18:30 CEST)?',
        pattern_status = 'detected' if pct >= 70 else 'unclear',
        n              = n_anomal_total,
        source_metric  = 'slot_annotation',
        evidence_str   = f'{n_anomal_evening}/{n_anomal_total} anomalous events in evening slot ({pct}%). Morning: {n_anomal_morning}.',
        mechanism      = ('Carnegie daily cycle: peak GEC activity ~19 UTC. '
                          'Evening slot (18:30 CEST = 16:30 UTC) captures the rising edge; '
                          'night slot (22:30 CEST = 20:30 UTC) captures post-peak.'),
        next_step      = 'More snapshots. Use midday slot (12:30) as control.',
        priority       = 'high',
        category       = 'core',
    ))

# H3 — L2→L3 Paradox
if pearson_l2_l3 < -0.1:
    factory_candidates.append(make_candidate(
        id             = 'H3',
        type           = 'structural_paradox',
        question       = 'Do L2 (weekly trend) and L3 (daily rhythm) operate on different timescales?',
        pattern_status = 'detected' if pearson_l2_l3 < -0.2 else 'unclear',
        n              = len(snaps),
        source_metric  = 'pearson_L2_L3',
        pearson        = round(pearson_l2_l3, 4),
        evidence_str   = (f'Pearson L2 vs L3 = {pearson_l2_l3:+.4f}. '
                          f'L2 trend={l2_trend:+.5f}/snap, L3 trend={l3_trend:+.5f}/snap. '
                          f'Gap trend={gap_trend:+.5f}/snap.'),
        mechanism      = 'L2 = SST/ENSO (weekly timescale), L3 = convective activity (daily timescale)',
        next_step      = 'Check lagged correlation L2(t) vs L3(t+24h/48h/7d).',
        priority       = 'high',
        category       = 'core',
    ))

# H4 — Carnegie amplitude modulator
if 'correlations' in carnegie_amplitude:
    top_var, top_r = max(carnegie_amplitude['correlations'].items(), key=lambda x: abs(x[1]))
    if abs(top_r) > 0.3:
        factory_candidates.append(make_candidate(
            id             = 'H4',
            type           = 'amplitude_modulator',
            question       = f'Does {top_var} modulate the Carnegie amplitude (L5 evening)?',
            pattern_status = 'detected' if abs(top_r) > 0.5 else 'unclear',
            n              = carnegie_amplitude['n_evenings'],
            source_metric  = f'pearson_L5evening_vs_{top_var}',
            pearson        = round(top_r, 4),
            evidence_str   = (f'Pearson L5_evening vs {top_var} = {top_r:+.3f} '
                              f'over {carnegie_amplitude["n_evenings"]} evenings.'),
            mechanism      = 'Thunderstorm activity (L3) drives GEC generator → L5 amplitude',
            next_step      = 'More evening snapshots. Compare all candidate variables in a table.',
            priority       = 'high',
            category       = 'core',
        ))

# H_op_cross_layer — transition dynamics
if operator_analysis.get('operator_dl3_correlation'):
    cross_r = operator_analysis['operator_dl3_correlation'].get('cross_layer_activation_operator')
    if cross_r is not None:
        factory_candidates.append(make_candidate(
            id             = 'H_op_cross_layer',
            type           = 'operator_predictor',
            question       = 'Is cross_layer_activation_operator a precursor for ΔL3 activation?',
            pattern_status = 'detected' if abs(cross_r) > 0.4 else 'unclear',
            n              = len(pairs_with_ops),
            source_metric  = 'cross_layer_activation_operator',
            pearson        = round(cross_r, 4),
            evidence_str   = f'Pearson cross_layer_activation vs ΔL3 = {cross_r:+.3f} (n={len(pairs_with_ops)}).',
            mechanism      = 'Gap between prepared and not-yet-activated layer as activation tension',
            next_step      = 'More operator snapshots. Check lead-lag at t+1 and t+2.',
            priority       = 'high',
            category       = 'core',
        ))

# ── DIAGNOSTIC CANDIDATES ────────────────────────────────────

# H6 — Persistent background tags
all_tags_flat = [t for s in snaps for t in s.get('event_tags', [])]
tag_freq_all  = Counter(all_tags_flat)
persistent    = [t for t, c in tag_freq_all.items()
                 if c == len(snaps) and not t.startswith('state_')]
if persistent:
    factory_candidates.append(make_candidate(
        id             = 'H6',
        type           = 'tag_hygiene',
        question       = 'Which tags are seasonal background and not actual activation signals?',
        pattern_status = 'detected',
        n              = len(snaps),
        source_metric  = 'event_tag_frequency',
        evidence_str   = f'Present in 100% of all snapshots: {", ".join(persistent)}.',
        mechanism      = 'Permanently present tags have no discriminative value for state prediction',
        next_step      = 'Finalize baseline_tags in Layer 7. Evaluate signal_tags separately.',
        priority       = 'medium',
        category       = 'diagnostic',
    ))

# H_cavity — Cavity Gate outcome pattern
if cavity_events:
    n_precursor = outcome_counts.get('cavity_shift_precursor', 0)
    n_failed    = outcome_counts.get('cavity_shift_failed', 0)
    n_total_cav = len(cavity_events)
    factory_candidates.append(make_candidate(
        id             = 'H_cavity',
        type           = 'precursor_pattern',
        question       = 'Does cavity_condition_shift_state more often precede activation than failure?',
        pattern_status = 'detected' if n_precursor > n_failed else 'unclear',
        n              = n_total_cav,
        source_metric  = 'cavity_gate_outcome',
        evidence_str   = (f'{n_total_cav} cavity events: '
                          f'precursor={n_precursor}, failed={n_failed}, '
                          f'neutral={outcome_counts.get("cavity_shift_neutral", 0)}.'),
        mechanism      = 'Cavity shift as electromagnetic precursor',
        next_step      = 'Collect more cavity events. Test L4/L6 coupling strength as threshold.',
        priority       = 'medium',
        category       = 'diagnostic',
    ))

# ── EXPLORATORY LEAD-LAG POOL ────────────────────────────────

if operator_analysis.get('lead_lag'):
    for key, info in operator_analysis['lead_lag'].items():
        r   = info['pearson']
        n_l = info['n_pairs']
        if abs(r) < 0.4: continue   # keep only interesting ones
        factory_candidates.append(make_candidate(
            id             = f'H_leadlag_{key}',
            type           = 'lead_lag_signal',
            question       = f'Does {key.split("__")[0]}(t) predict the value of {key.split("__")[1]}?',
            pattern_status = 'detected' if abs(r) > 0.5 else 'unclear',
            n              = n_l,
            source_metric  = key,
            pearson        = round(r, 4),
            lag            = info['lag_snapshots'],
            evidence_str   = (f'Pearson={r:+.3f}, n={n_l}, '
                              f'lag={info["lag_snapshots"]} snapshots (~{info["lag_snapshots"]*6}h).'),
            mechanism      = None,   # must be added manually
            next_step      = 'Identify mechanism. Wait for n >= 10.',
            priority       = 'low',
            category       = 'exploratory_leadlag',
        ))

# ── Load & save registry ─────────────────────────────────────

if os.path.exists(REGISTRY_FILE):
    with open(REGISTRY_FILE, encoding='utf-8') as f:
        registry = json.load(f)
else:
    registry = {}

new_count = 0
for c in factory_candidates:
    hid = c['id']
    # Carry over review fields from registry (preserve manual decisions)
    if hid in registry:
        c['review_status']          = registry[hid].get('review_status', 'unreviewed')
        c['include_in_report']      = registry[hid].get('include_in_report', False)
        c['include_in_model_logic'] = registry[hid].get('include_in_model_logic', False)
        c['reviewer_notes']         = registry[hid].get('reviewer_notes', '')
        registry[hid]['last_updated'] = RUN_TS
    else:
        c['reviewer_notes'] = ''
        registry[hid] = {
            'review_status':          'unreviewed',
            'include_in_report':      False,
            'include_in_model_logic': False,
            'reviewer_notes':         '',
            'first_seen':             RUN_TS,
        }
        new_count += 1

    # Re-evaluate promotion block after review status update
    c['promotion_blocked_reason'] = promotion_blocked_reason(
        c['n'], c['pattern_status'], c['review_status']
    )

    # Save as individual JSON file
    fp = os.path.join(CANDIDATES_DIR, f'{hid}.json')
    with open(fp, 'w', encoding='utf-8') as f:
        json.dump(c, f, indent=2, ensure_ascii=False)

with open(REGISTRY_FILE, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2, ensure_ascii=False)

# ── Output ───────────────────────────────────────────────────

categories = ['core', 'diagnostic', 'exploratory_leadlag']
cat_labels  = {
    'core':                '── CORE CANDIDATES ──',
    'diagnostic':          '── DIAGNOSTIC CANDIDATES ──',
    'exploratory_leadlag': '── EXPLORATORY LEAD-LAG POOL ──',
}
ev_icon = {
    'exploratory_signal': '🔬',
    'testable':           '🧪',
    'moderate_evidence':  '📊',
    'robust_candidate':   '✔️',
    'unknown':            '❓',
}
pat_icon = {'detected': '✅', 'unclear': '⚠️', 'absent': '❌'}

print('HYPOTHESIS FACTORY v2')
print('=' * 78)
print(f'  Candidates total: {len(factory_candidates)}  |  New: {new_count}  |  Registry: {REGISTRY_FILE}')

for cat in categories:
    group = [c for c in factory_candidates if c['category'] == cat]
    if not group: continue
    print(f'\n  {cat_labels[cat]}')
    for c in sorted(group, key=lambda x: {'high':0,'medium':1,'low':2,'quarantine':3}[x['priority']]):
        pat   = pat_icon.get(c['pattern_status'], '?')
        ev    = ev_icon.get(c['evidence_level'], '?')
        n_str = f'n={c["n"]}' if c['n'] is not None else 'n=?'
        r_str = f'r={c["pearson"]:+.3f}' if c.get('pearson') is not None else ''
        blocked = f'  🔒 {c["promotion_blocked_reason"]}' if c['promotion_blocked_reason'] else ''
        print(f'  {pat} {ev} [{c["priority"]:<6}] {c["id"]:<40} {n_str:<6} {r_str}{blocked}')
        if c['mechanism']:
            print(f'      └─ {c["mechanism"][:75]}')

print(f'\n  Legend: ✅=detected ⚠️=unclear  🔬=exploratory 🧪=testable 📊=moderate ✔️=robust')

HYPOTHESIS FACTORY v2
  Candidates total: 4  |  New: 0  |  Registry: ../research/test_hypothesis_registry.json

  ── CORE CANDIDATES ──
  ✅ 🔬 [high  ] H1                                       n=3      🔒 not_manually_reviewed
      └─ Carnegie daily cycle: peak GEC activity ~19 UTC. Evening slot (18:30 CEST =
  ⚠️ 🧪 [high  ] H3                                       n=21   r=-0.140  🔒 not_manually_reviewed
      └─ L2 = SST/ENSO (weekly timescale), L3 = convective activity (daily timescale

  ── DIAGNOSTIC CANDIDATES ──
  ✅ 🧪 [medium] H6                                       n=21     🔒 not_manually_reviewed
      └─ Permanently present tags have no discriminative value for state prediction
  ⚠️ 🔬 [medium] H_cavity                                 n=7      🔒 not_manually_reviewed
      └─ Cavity shift as electromagnetic precursor

  Legend: ✅=detected ⚠️=unclear  🔬=exploratory 🧪=testable 📊=moderate ✔️=robust
